In [1]:
import pandas as pd 
from bs4 import BeautifulSoup
import requests

In [17]:
url = 'https://remoteok.com/'


HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}

try:
    r_ = requests.get(url, headers=HEADERS, allow_redirects=False, timeout=10)
    print(r_.status_code)
except Exception as e:
    print(f"Error: {e}")


200


In [18]:
print(r_.text)

<!doctype html><html lang="en" class="   pageType-frontpage  remoteok    minimize-header   catch-emails-enabled">	<head>
			


					<link rel="stylesheet" href="/global.css?1780743778">
					<script>
													var userIsAdmin=false;
											</script>
					<meta charset="UTF-8">
					<title>Remote Jobs in Programming, Design, Sales and more #OpenSalaries</title>
					<meta name="description" content="Looking for a remote job? Remote OK® is the #1 Remote Job Platform and has 1,134,166+ remote jobs as a Developer, Designer, Copywriter, Customer Support Rep, Sales Professional, Project Manager and more! Find a career where you can work remotely from anywhere.">
					<meta name="msvalidate.01" content="5E0F32F90CCE5417B26666C526F8B071" />
											<meta name="viewport" content="width=device-width, initial-scale=1">
										
					
					<meta http-equiv="Content-Language" content="en-US">
					<link rel="search" type="application/opensearchdescription+xml" title="Remote OK" href="/o

In [20]:
soup = BeautifulSoup(r_.text, "html.parser")

In [41]:
# Job rows
job_rows = soup.find("tr", {"data-offset":"9"})


In [42]:
job_rows

In [48]:
!playwright install

169.2 MiB [                    ] 0% 0.0s169.2 MiB [                    ] 0% 226.9s169.2 MiB [                    ] 0% 527.4s169.2 MiB [                    ] 0% 1144.3s169.2 MiB [                    ] 0% 837.4s169.2 MiB [                    ] 0% 788.3s169.2 MiB [                    ] 0% 682.9s169.2 MiB [                    ] 0% 730.4s169.2 MiB [                    ] 0% 628.2s169.2 MiB [                    ] 0% 605.9s169.2 MiB [                    ] 0% 579.9s169.2 MiB [                    ] 0% 565.7s169.2 MiB [                    ] 0% 587.8s169.2 MiB [                    ] 0% 493.9s169.2 MiB [                    ] 0% 460.9s169.2 MiB [                    ] 0% 430.7s169.2 MiB [                    ] 0% 405.3s169.2 MiB [                    ] 0% 384.5s169.2 MiB [                    ] 0% 409.2s169.2 MiB [                    ] 0% 342.2s169.2 MiB [                    ] 0% 324.8s169.2 MiB [                    ] 0% 303.1s169.2 MiB [                    ] 0% 347.5s169.2 MiB [                    ] 0%

In [53]:
import asyncio
from playwright.async_api import async_playwright

async def run():

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)

        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
        )

        page = await context.new_page()

        print("Connecting to Remote OK...")
        await page.goto("https://remoteok.com/", wait_until="networkidle")
        
        cdp_session = await page.context.new_cdp_session(page)

        print("Compiling website assest into HTML")
        snapshot = await cdp_session.send('Page.captureSnapshot', {'format': 'mhtml'})

        mhtml_content = snapshot['data']

        file = "remoteok_complete.mhtml"
        with open(file, "w", encoding="utf-8") as f:
            f.write(mhtml_content)

        print(f"Success {file}")

        await context.close()
        browser.close()

# In a Jupyter notebook, execute the async function using top-level await
await run()

Connecting to Remote OK...
Compiling website assest into HTML
Success remoteok_complete.mhtml


/var/folders/md/v_wl37p57v5_mnng5bdt123m0000gn/T/ipykernel_74524/3429188771.py:32: RuntimeWarning: coroutine 'Browser.close' was never awaited
  browser.close()


In [ ]:
import asyncio
from playwright.async_api import async_playwright

async def run():
    async with async_playwright() as p:

        browser = await p.chromium.launch(headless=False)

        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
        )

        page = await context.new_page()

        print("Connecting to Remote Ok...")
        await page.goto("https://remoteok.com/", wait_until="domcontentloaded")

        
        print("Scrolling to the absolute bottom of the page...")

        previous_height = await page.evaluate("document.body.scrollHeight")
        while True:
            await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")

            await page.wait_for_timeout(2000)

            new_height = await page.evaluate("document.body.scrollHeight")
            if new_height == previous_height:
                print("Reached the bottom of the page. No more jobs loading")
                break
            previous_height = new_height

        cdp_session = await page.context.new_cdp_session(page)

        print("Compiling website assets into HTML snapshot...")
        snapshot = await cdp_session.send('Page.captureSnapshot', {'format': 'mhtml'})

        mhtml_content = snapshot['data']

        file = "remoteok_complete.mhtml"
        with open(file, "w", encoding="utf-8") as f:
            f.write(mhtml_content)

        print(f"Success: {file}")

        await context.close()
        await browser.close()

await run()

In [57]:
import email
import re

path = '/Users/apple/hiring/remoteok_complete.mhtml'

with open(path, "r") as f:
    msg = email.message_from_string(f.read())


html_content = ""
for part in msg.walk():
    if part.get_content_type() == "text/html":
        html_content = part.get_payload(decode=True).decode("utf-8", errors="ignore")
        break


soup = BeautifulSoup(html_content, "html.parser")
print(f"Successfully loaded and parsed HTML document!")
print(f"Page Target Title: {soup.title.text if soup.title else 'No title found'}")

Successfully loaded and parsed HTML document!
Page Target Title: Remote Jobs in Programming, Design, Sales and more #OpenSalaries


In [60]:
job_rows = soup.find_all("tr", id=re.compile(r"^job-\d+$"))

records = []

for row in job_rows:
    job_id = row.get("data-id", "").strip()
    slug = row.get("data-slug", "").strip()
    epoch = row.get("data-epoch", "").strip()
    apply_path = row.get("data-href", "").strip()


    # Title
    title_tag = row.find(itemprop="title")
    title = title_tag.get_text(strip=True) if title_tag else None

    # Company 
    company_tag = row.find(itemprop="name")
    company = company_tag.get_text(strip=True) if company_tag else None

    # Location
    loc_tag = row.find("div", class_="location")
    location = loc_tag.get_text(strip=True) if loc_tag else None

    # Salary
    sal_tag = row.find("div", class_="salary")
    salary_raw = sal_tag.get_text(strip=True) if sal_tag else None

    # Tags
    tag_divs = row.find_all("div", class_=re.compile(r"\btag\b"))
    tags = [t.get_text(strip=True) for t in tag_divs if t.get_text(strip=True)]

    # Posted datetime
    time_tag = row.find("time")
    posted_at = time_tag.get("datetime") if time_tag else None


    records.append(
        {
            "job_id": job_id,
            "slug": slug,
            "title": title,
            "company": company,
            "location": location,
            "salary_raw": salary_raw,
            "tags": tags,
            "posted_at": posted_at,
            "epoch": int(epoch) if epoch.isdigit() else None,
            "aply_url": f"https://remoteok.com{apply_path}" if apply_path else None
        }
    )


records

[{'job_id': '1132895',
  'slug': 'remote-support-engineer-level-1132895',
  'title': 'Support Engineer',
  'company': 'Level',
  'location': '🌏 Worldwide',
  'salary_raw': '💰 $60k - $90k',
  'tags': ['Customer Support', 'Technical'],
  'posted_at': '2026-06-05T18:33:48+00:00',
  'epoch': 1780684428,
  'aply_url': 'https://remoteok.com/remote-jobs/remote-support-engineer-level-1132895'},
 {'job_id': '1131368',
  'slug': 'remote-sales-reps-wanted-for-high-ticket-finance-sales-fse-1131368',
  'title': 'SALES REPS WANTED FOR HIGH TICKET FINANCE SALES',
  'company': 'FSE',
  'location': '🌏 Worldwide',
  'salary_raw': '💰 $60k - $200k',
  'tags': ['Sales'],
  'posted_at': '2026-06-27T00:00:09+00:00',
  'epoch': 1782518409,
  'aply_url': 'https://remoteok.com/remote-jobs/remote-sales-reps-wanted-for-high-ticket-finance-sales-fse-1131368'},
 {'job_id': '1132435',
  'slug': 'remote-client-delivery-manager-storyteller-1132435',
  'title': 'Client Delivery Manager',
  'company': 'Storyteller',
  '

In [61]:
df = pd.DataFrame(records)

In [62]:
df

,job_id,slug,title,company,location,salary_raw,tags,posted_at,epoch,aply_url
0,1132895,remote-support-engineer-level-1132895,Support Engineer,Level,🌏 Worldwide,💰 $60k - $90k,"[Customer Support, Technical]",2026-06-05T18:33:48+00:00,1780684428,https://remoteok.com/remote-jobs/remote-suppor...
1,1131368,remote-sales-reps-wanted-for-high-ticket-finan...,SALES REPS WANTED FOR HIGH TICKET FINANCE SALES,FSE,🌏 Worldwide,💰 $60k - $200k,[Sales],2026-06-27T00:00:09+00:00,1782518409,https://remoteok.com/remote-jobs/remote-sales-...
2,1132435,remote-client-delivery-manager-storyteller-113...,Client Delivery Manager,Storyteller,🇧🇷 Brazil,💰 $40k - $80k,"[Work from Home, AI, SaaS, Executive]",2026-06-27T00:00:02+00:00,1782518402,https://remoteok.com/remote-jobs/remote-client...
3,1134014,remote-mid-senior-ai-cinematic-video-editor-ev...,Mid Senior AI Cinematic Video Editor,EverAI,🌍 Africa,💰 $30k - $100k,"[Design, AI, Other, Art Direction]",2026-06-24T09:04:44+00:00,1782291884,https://remoteok.com/remote-jobs/remote-mid-se...
4,1133914,remote-head-of-public-understanding-nope-1133914,Head of Public Understanding,NOPE,⛩ Asia,💰 $100k - $150k,"[Marketing, Communications, Content Writing, A...",2026-06-23T03:37:28+00:00,1782185848,https://remoteok.com/remote-jobs/remote-head-o...
5,1134159,remote-product-growth-expert-acquisition-afirm...,Product Growth Expert Acquisition Afirmativa p...,Neon Pagamentos,Remoto,NaN,"[Marketing, Python, C, Node, Excel]",2026-06-27T00:00:04+00:00,1782518404,https://remoteok.com/remote-jobs/remote-produc...
6,1134171,remote-property-maintenance-amp-operations-coo...,Property Maintenance & Operations Coordinator,Dart,🇰🇾 Cayman Islands,NaN,"[Sys Admin, Edu, Non Tech, InfoSec, Technical,...",2026-06-26T20:23:35+00:00,1782505415,https://remoteok.com/remote-jobs/remote-proper...
7,1134164,remote-common-energy-office-of-energy-amp-sust...,Common Energy,Office of Energy & Sustainability,🌏 Worldwide,NaN,"[Supervisor, Executive, Sys Admin, Customer Su...",2026-06-26T20:20:01+00:00,1782505201,https://remoteok.com/remote-jobs/remote-common...
8,1134172,remote-people-operations-specialist-vulncheck-...,People Operations Specialist,VulnCheck,🇺🇸 United States,NaN,"[HR, InfoSec, Coordinator, Customer Support, M...",2026-06-26T19:57:38+00:00,1782503858,https://remoteok.com/remote-jobs/remote-people...
9,1134158,remote-frontend-developer-junior-universia-per...,Frontend Developer Junior,Universia Perú,🇵🇪 Peru,NaN,"[Front End, Developer, Web Developer, JavaScri...",2026-06-26T19:18:48+00:00,1782501528,https://remoteok.com/remote-jobs/remote-fronte...


In [68]:
# Cleaning Process

cleaned_df = df[df["title"].notna() & (df["title"] != "")].copy()


def parse_salary(s):
    if not isinstance(s, str):
        return None, None
    s = s.replace(",", "").replace("$", "").replace("💰", "").strip()
    nums = re.findall(r"(\d+(?:\.\d+)?)\s*k?", s, re.IGNORECASE)
    multipliers = [1000 if "k" in tok.lower() else 1
        for tok in re.findall(r"(\d+(?:\.\d+)?k?)", s, re.IGNORECASE)]

    values = []
    for n, m in zip(nums, multipliers):
        values.append(int(float(n) * m))

    k_flags = [bool(re.search(r"\d+k", tok, re.IGNORECASE))
                   for tok in re.findall(r"\d+\.?\d*k?", s, re.IGNORECASE)]
    values = []
    for tok in re.findall(r"\d+\.?\d*k?", s, re.IGNORECASE):
        num = float(re.search(r"[\d.]+", tok).group())
        if "k" in tok.lower():
            num *= 1000
        values.append(int(num))
    if len(values) == 0:
        return None, None
    if len(values) == 1:
        return values[0], values[0]
    return values[0], values[1]

def strip_emoji(s):
    if not isinstance(s, str):
        return s
    return re.sub(r"^[\U00010000-\U0010ffff\s]+", "", s).strip()

cleaned_df["location"] = cleaned_df["location"].apply(strip_emoji)

cleaned_df[["salary_min", "salary_max"]] = cleaned_df["salary_raw"].apply(
    lambda s: pd.Series(parse_salary(s))
)

In [69]:
cleaned_df

,job_id,slug,title,company,location,salary_raw,tags,posted_at,epoch,aply_url,salary_min,salary_max
0,1132895,remote-support-engineer-level-1132895,Support Engineer,Level,Worldwide,💰 $60k - $90k,"[Customer Support, Technical]",2026-06-05T18:33:48+00:00,1780684428,https://remoteok.com/remote-jobs/remote-suppor...,60000.0,90000.0
1,1131368,remote-sales-reps-wanted-for-high-ticket-finan...,SALES REPS WANTED FOR HIGH TICKET FINANCE SALES,FSE,Worldwide,💰 $60k - $200k,[Sales],2026-06-27T00:00:09+00:00,1782518409,https://remoteok.com/remote-jobs/remote-sales-...,60000.0,200000.0
2,1132435,remote-client-delivery-manager-storyteller-113...,Client Delivery Manager,Storyteller,Brazil,💰 $40k - $80k,"[Work from Home, AI, SaaS, Executive]",2026-06-27T00:00:02+00:00,1782518402,https://remoteok.com/remote-jobs/remote-client...,40000.0,80000.0
3,1134014,remote-mid-senior-ai-cinematic-video-editor-ev...,Mid Senior AI Cinematic Video Editor,EverAI,Africa,💰 $30k - $100k,"[Design, AI, Other, Art Direction]",2026-06-24T09:04:44+00:00,1782291884,https://remoteok.com/remote-jobs/remote-mid-se...,30000.0,100000.0
4,1133914,remote-head-of-public-understanding-nope-1133914,Head of Public Understanding,NOPE,⛩ Asia,💰 $100k - $150k,"[Marketing, Communications, Content Writing, A...",2026-06-23T03:37:28+00:00,1782185848,https://remoteok.com/remote-jobs/remote-head-o...,100000.0,150000.0
5,1134159,remote-product-growth-expert-acquisition-afirm...,Product Growth Expert Acquisition Afirmativa p...,Neon Pagamentos,Remoto,NaN,"[Marketing, Python, C, Node, Excel]",2026-06-27T00:00:04+00:00,1782518404,https://remoteok.com/remote-jobs/remote-produc...,NaN,NaN
6,1134171,remote-property-maintenance-amp-operations-coo...,Property Maintenance & Operations Coordinator,Dart,Cayman Islands,NaN,"[Sys Admin, Edu, Non Tech, InfoSec, Technical,...",2026-06-26T20:23:35+00:00,1782505415,https://remoteok.com/remote-jobs/remote-proper...,NaN,NaN
7,1134164,remote-common-energy-office-of-energy-amp-sust...,Common Energy,Office of Energy & Sustainability,Worldwide,NaN,"[Supervisor, Executive, Sys Admin, Customer Su...",2026-06-26T20:20:01+00:00,1782505201,https://remoteok.com/remote-jobs/remote-common...,NaN,NaN
8,1134172,remote-people-operations-specialist-vulncheck-...,People Operations Specialist,VulnCheck,United States,NaN,"[HR, InfoSec, Coordinator, Customer Support, M...",2026-06-26T19:57:38+00:00,1782503858,https://remoteok.com/remote-jobs/remote-people...,NaN,NaN
9,1134158,remote-frontend-developer-junior-universia-per...,Frontend Developer Junior,Universia Perú,Peru,NaN,"[Front End, Developer, Web Developer, JavaScri...",2026-06-26T19:18:48+00:00,1782501528,https://remoteok.com/remote-jobs/remote-fronte...,NaN,NaN


In [70]:
cleaned_df

,job_id,slug,title,company,location,salary_raw,tags,posted_at,epoch,aply_url,salary_min,salary_max
0,1132895,remote-support-engineer-level-1132895,Support Engineer,Level,Worldwide,💰 $60k - $90k,"[Customer Support, Technical]",2026-06-05T18:33:48+00:00,1780684428,https://remoteok.com/remote-jobs/remote-suppor...,60000.0,90000.0
1,1131368,remote-sales-reps-wanted-for-high-ticket-finan...,SALES REPS WANTED FOR HIGH TICKET FINANCE SALES,FSE,Worldwide,💰 $60k - $200k,[Sales],2026-06-27T00:00:09+00:00,1782518409,https://remoteok.com/remote-jobs/remote-sales-...,60000.0,200000.0
2,1132435,remote-client-delivery-manager-storyteller-113...,Client Delivery Manager,Storyteller,Brazil,💰 $40k - $80k,"[Work from Home, AI, SaaS, Executive]",2026-06-27T00:00:02+00:00,1782518402,https://remoteok.com/remote-jobs/remote-client...,40000.0,80000.0
3,1134014,remote-mid-senior-ai-cinematic-video-editor-ev...,Mid Senior AI Cinematic Video Editor,EverAI,Africa,💰 $30k - $100k,"[Design, AI, Other, Art Direction]",2026-06-24T09:04:44+00:00,1782291884,https://remoteok.com/remote-jobs/remote-mid-se...,30000.0,100000.0
4,1133914,remote-head-of-public-understanding-nope-1133914,Head of Public Understanding,NOPE,⛩ Asia,💰 $100k - $150k,"[Marketing, Communications, Content Writing, A...",2026-06-23T03:37:28+00:00,1782185848,https://remoteok.com/remote-jobs/remote-head-o...,100000.0,150000.0
5,1134159,remote-product-growth-expert-acquisition-afirm...,Product Growth Expert Acquisition Afirmativa p...,Neon Pagamentos,Remoto,NaN,"[Marketing, Python, C, Node, Excel]",2026-06-27T00:00:04+00:00,1782518404,https://remoteok.com/remote-jobs/remote-produc...,NaN,NaN
6,1134171,remote-property-maintenance-amp-operations-coo...,Property Maintenance & Operations Coordinator,Dart,Cayman Islands,NaN,"[Sys Admin, Edu, Non Tech, InfoSec, Technical,...",2026-06-26T20:23:35+00:00,1782505415,https://remoteok.com/remote-jobs/remote-proper...,NaN,NaN
7,1134164,remote-common-energy-office-of-energy-amp-sust...,Common Energy,Office of Energy & Sustainability,Worldwide,NaN,"[Supervisor, Executive, Sys Admin, Customer Su...",2026-06-26T20:20:01+00:00,1782505201,https://remoteok.com/remote-jobs/remote-common...,NaN,NaN
8,1134172,remote-people-operations-specialist-vulncheck-...,People Operations Specialist,VulnCheck,United States,NaN,"[HR, InfoSec, Coordinator, Customer Support, M...",2026-06-26T19:57:38+00:00,1782503858,https://remoteok.com/remote-jobs/remote-people...,NaN,NaN
9,1134158,remote-frontend-developer-junior-universia-per...,Frontend Developer Junior,Universia Perú,Peru,NaN,"[Front End, Developer, Web Developer, JavaScri...",2026-06-26T19:18:48+00:00,1782501528,https://remoteok.com/remote-jobs/remote-fronte...,NaN,NaN
